# ROPG-KD Retriever Training

Stage 1 of Simurgh's two-stage training: knowledge-distillation fine-tuning of the Qwen3-Embedding-0.6B encoder, distilling from the offline LLM-judge teacher scores in `data/ropg_kd/{train,val}.jsonl` (see `notebooks/gen_ropg_data.ipynb`); checkpoints are selected each epoch on validation Recall@K / MRR per persona, not on training or validation KD loss.

**Kaggle setup checklist**
1. Enable GPU accelerator (T4 x1 is enough; ~10-20 min for 5 epochs).
2. Enable internet access (the encoder is downloaded from Hugging Face).
3. Attach the `simurgh-data` dataset, **version 2 or later** (must contain `ropg_kd/`).
4. No API secrets needed — training makes no LLM calls.

**Colab setup checklist**
1. Set `RUNTIME = "colab"` in the Config cell below.
2. Upload `simurgh-data/` to Google Drive at `MyDrive/simurgh-data/` — must contain `ropg_kd/train.jsonl`, `ropg_kd/val.jsonl`.
3. Enable GPU accelerator (T4 × 1 is enough; ~10–20 min for 5 epochs).
4. No API secrets needed.
5. Checkpoints are saved directly to Google Drive — the final zip cell is skipped automatically.

In [ ]:
!pip install -q transformers accelerate

## Config

In [ ]:
import os

# Must be set before torch initializes CUDA; mitigates fragmentation OOMs by
# letting the allocator grow segments instead of hunting for contiguous blocks.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from pathlib import Path

# ── Runtime selector ─────────────────────────────────────────────────────────
# Set RUNTIME to match where you are running this notebook.
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"  # Colab only

# ── Paths ─────────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    DATASET_SLUG = "simurgh-data"
    DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
    OUTPUT_DIR = "/kaggle/working/ropg_kd_checkpoints"
elif RUNTIME == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_ROOT = GDRIVE_BASE
    OUTPUT_DIR = f"{GDRIVE_BASE}/ropg_kd_checkpoints"
else:  # local
    DATA_ROOT = "data"
    OUTPUT_DIR = "data/ropg_kd_checkpoints"

# ── Inline config (mirrors configs/train_ropg.yaml) ─────────────────────────
CFG = {
    "mode": "reader_kd",
    "format": "triplets",
    "data": {
        "train_data": f"{DATA_ROOT}/ropg_kd",
    },
    "embedder": {
        "model": "Qwen/Qwen3-Embedding-0.6B",
        "max_seq_length": 512,  # most chunks are <260 tokens; 8192 wastes O(n²) attention
    },
    "training": {
        "device": "cuda",  # cpu for local smoke runs
        "precision": "fp16",  # fp16 | bf16 | fp32 — fp16 for T4/P100, bf16 for A100+
        "epochs": 5,
        "batch_size": 8,  # groups per optimizer step (gradient accumulation)
        "lr": 2.0e-4,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "grad_clip": 1.0,
        "max_negatives": 4,
        "max_documents": 20,
        "temperature": 1.0,
    },
    "eval": {
        "top_k": 5,  # matches inference retrieval.top_k
        "eval_batch_size": 32,
        "best_metric": "recall@1",
    },
    "checkpoint_dir": OUTPUT_DIR,
    "seed": 42,
}

Path(CFG["checkpoint_dir"]).mkdir(parents=True, exist_ok=True)


## Core classes / personas

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    id: str
    split: str
    rendered: str


PERSONAS = {
    "crammer": Profile(
        id="crammer",
        split="train",
        rendered=(
            "A ninth-grader who finds the textbook hard to follow and has little background "
            "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
            "to score — but it must be spelled out simply, step by step, with examples."
        ),
    ),
    "scholar": Profile(
        id="scholar",
        split="train",
        rendered=(
            "A ninth-grader who reads dense material easily and has solid background on this "
            "topic. Wants to understand the underlying why and how, and the connections between "
            "ideas. Prefers a terse, high-level treatment without hand-holding or padding."
        ),
    ),
    "steady": Profile(
        id="steady",
        split="train",
        rendered=(
            "A capable ninth-grader with average background on this topic. Wants a correct "
            "answer with a brief justification, balanced toward exam needs. Does not need "
            "elaborate scaffolding, but does appreciate a one-line reason."
        ),
    ),
}


def render_profile(persona_id: str) -> str:
    return PERSONAS[persona_id].rendered


def train_personas() -> list:
    return [p for p in PERSONAS.values() if p.split == "train"]

In [ ]:
# ── Model & Dataset classes (from src/rl/ropg_kd.py) ─────────────────────────
from __future__ import annotations

import json
from collections import defaultdict
from typing import Any, Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from transformers import AutoModel


class QwenEmbeddingModel(nn.Module):
    """Qwen3-Embedding wrapper with mean pooling + L2 normalization."""

    def __init__(self, model_name: str, use_gradient_checkpointing: bool = True):
        super().__init__()
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        self.model = AutoModel.from_pretrained(
            model_name, trust_remote_code=True, torch_dtype=dtype
        )
        if use_gradient_checkpointing:
            self.model.gradient_checkpointing_enable()

    def forward(
        self, input_ids: torch.Tensor, attention_mask: torch.Tensor
    ) -> torch.Tensor:
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        summed = (hidden * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-9)
        emb = summed / denom
        return F.normalize(emb, p=2, dim=1)


class TripletDataset(Dataset):
    """Reads {query, positive, negatives: [...]} lines.
    If 'persona_id' is present, prepend the rendered profile to the query.
    """

    def __init__(self, jsonl_path: str, max_negatives: int = 4) -> None:
        self.items: List[Dict[str, Any]] = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line.strip())
                negs = obj.get("negatives", [])[:max_negatives]
                if not negs:
                    continue
                query = obj["query"]
                if "persona_id" in obj:
                    persona_text = render_profile(obj["persona_id"])
                    query = f"{persona_text}\n\n{query}"
                self.items.append(
                    {
                        "query": query,
                        "positive": obj["positive"],
                        "negatives": negs,
                    }
                )

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.items[idx]


class PairDataset(Dataset):
    """Reads {query, positive, negative} lines (cartesian-expanded).
    If 'persona_id' is present, prepend the rendered profile to the query.
    """

    def __init__(self, jsonl_path: str) -> None:
        self.items: List[Dict[str, str]] = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line.strip())
                query = obj["query"]
                if "persona_id" in obj:
                    persona_text = render_profile(obj["persona_id"])
                    query = f"{persona_text}\n\n{query}"
                self.items.append(
                    {
                        "query": query,
                        "positive": obj["positive"],
                        "negative": obj["negative"],
                    }
                )

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> Dict[str, str]:
        return self.items[idx]


class ScoredDataset(Dataset):
    """Reads scored lines for KL-distillation.

    Supports two JSONL formats:
      1. Old: {"query": "...", "document": "...", "score": 0.5}
      2. New: {"query": "...", "persona_id": "...", "docs": [{"chunk_id": "...", "text": "...", "teacher_score": 0.3}, ...]}

    If 'persona_id' is present, prepend the rendered profile to the query.
    """

    def __init__(self, jsonl_path: str, max_documents: int = 20) -> None:
        grouped: Dict[str, List[Tuple[str, float]]] = {}
        order: List[str] = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line.strip())
                q = obj["query"]
                if "persona_id" in obj:
                    persona_text = render_profile(obj["persona_id"])
                    q = f"{persona_text}\n\n{q}"
                if q not in grouped:
                    grouped[q] = []
                    order.append(q)

                if "docs" in obj:
                    for doc in obj["docs"]:
                        grouped[q].append((doc["text"], float(doc["teacher_score"])))
                elif "document" in obj and "score" in obj:
                    grouped[q].append((obj["document"], float(obj["score"])))
                else:
                    continue  # unknown format

        self.items: List[Dict[str, Any]] = []
        for q in order:
            docs_scores = grouped[q][:max_documents]
            if len(docs_scores) < 2:
                continue
            docs, scores = zip(*docs_scores)
            self.items.append(
                {
                    "query": q,  # already modified with persona
                    "documents": list(docs),
                    "scores": list(scores),
                }
            )

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.items[idx]


def collate_triplets(
    batch: List[Dict[str, Any]],
    tokenizer,
    max_length: int,
) -> Dict[str, torch.Tensor]:
    queries = [x["query"] for x in batch]
    positives = [x["positive"] for x in batch]
    max_negs = max(len(x["negatives"]) for x in batch)

    negs_flat: List[str] = []
    for x in batch:
        negs_flat.extend(x["negatives"])
        negs_flat.extend([""] * (max_negs - len(x["negatives"])))

    q = tokenizer(
        queries,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    p = tokenizer(
        positives,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    n = tokenizer(
        negs_flat,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    B = len(batch)
    return {
        "q_ids": q["input_ids"],
        "q_mask": q["attention_mask"],
        "p_ids": p["input_ids"],
        "p_mask": p["attention_mask"],
        "n_ids": n["input_ids"].view(B, max_negs, -1),
        "n_mask": n["attention_mask"].view(B, max_negs, -1),
    }


def collate_pairs(
    batch: List[Dict[str, str]],
    tokenizer,
    max_length: int,
) -> Dict[str, torch.Tensor]:
    triplet_batch = [
        {"query": x["query"], "positive": x["positive"], "negatives": [x["negative"]]}
        for x in batch
    ]
    return collate_triplets(triplet_batch, tokenizer, max_length)


def collate_scored(
    batch: List[Dict[str, Any]],
    tokenizer,
    max_length: int,
) -> Dict[str, torch.Tensor]:
    queries = [x["query"] for x in batch]
    max_docs = max(len(x["documents"]) for x in batch)

    docs_flat: List[str] = []
    gold_scores: List[List[float]] = []
    doc_mask: List[List[bool]] = []
    for x in batch:
        n = len(x["documents"])
        docs_flat.extend(x["documents"])
        docs_flat.extend([""] * (max_docs - n))
        padded_scores = list(x["scores"]) + [-1e9] * (max_docs - n)
        gold_scores.append(padded_scores)
        doc_mask.append([True] * n + [False] * (max_docs - n))

    q = tokenizer(
        queries,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    d = tokenizer(
        docs_flat,
        max_length=max_length,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    B = len(batch)
    return {
        "q_ids": q["input_ids"],
        "q_mask": q["attention_mask"],
        "d_ids": d["input_ids"].view(B, max_docs, -1),
        "d_mask": d["attention_mask"].view(B, max_docs, -1),
        "gold_scores": torch.tensor(gold_scores, dtype=torch.float32),
        "doc_mask": torch.tensor(doc_mask, dtype=torch.bool),
    }


def _move(
    batch: Dict[str, torch.Tensor], device: torch.device
) -> Dict[str, torch.Tensor]:
    return {k: v.to(device) for k, v in batch.items()}


In [ ]:
# ── Loss functions & training loops (from src/rl/ropg_kd.py) ────────────────

from tqdm.auto import tqdm

def mnrl_loss(
    query_emb: torch.Tensor,
    pos_emb: torch.Tensor,
    neg_emb: torch.Tensor,
    temperature: float = 0.05,
) -> torch.Tensor:
    q = query_emb / temperature
    p = pos_emb / temperature
    n = neg_emb / temperature
    pos_score = (q * p).sum(dim=-1, keepdim=True)
    neg_scores = torch.bmm(n, q.unsqueeze(-1)).squeeze(-1)
    all_scores = torch.cat([pos_score, neg_scores], dim=-1)
    labels = torch.zeros(q.size(0), dtype=torch.long, device=q.device)
    return F.cross_entropy(all_scores, labels)


def kd_loss(
    student_scores: torch.Tensor,
    gold_scores: torch.Tensor,
    temperature: float = 1.0,
) -> torch.Tensor:
    student_logprobs = F.log_softmax(student_scores / temperature, dim=-1)
    gold_probs = F.softmax(gold_scores / temperature, dim=-1)
    return F.kl_div(student_logprobs, gold_probs, reduction="batchmean")


# ── Training loops ───────────────────────────────────────────────────────────

def train_mnrl_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    temperature,
    use_amp,
):
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        batch = _move(batch, device)
        q_ids, q_mask = batch["q_ids"], batch["q_mask"]
        p_ids, p_mask = batch["p_ids"], batch["p_mask"]
        n_ids, n_mask = batch["n_ids"], batch["n_mask"]
        B, N, L = n_ids.shape

        with torch.autocast(device_type="cuda", enabled=use_amp, dtype=torch.bfloat16):
            q_emb = model(q_ids, q_mask)
            p_emb = model(p_ids, p_mask)
            n_flat = model(n_ids.view(B * N, L), n_mask.view(B * N, L)).view(B, N, -1)
            loss = mnrl_loss(q_emb, p_emb, n_flat, temperature)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total += loss.item()
        n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total / max(n, 1)


def train_kd_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device,
    temperature,
    use_amp,
):
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(loader, desc="train", leave=False)
    for batch in pbar:
        batch = _move(batch, device)
        q_ids, q_mask = batch["q_ids"], batch["q_mask"]
        d_ids, d_mask = batch["d_ids"], batch["d_mask"]
        gold = batch["gold_scores"]
        B, D, L = d_ids.shape

        with torch.autocast(device_type="cuda", enabled=use_amp, dtype=torch.bfloat16):
            q_emb = model(q_ids, q_mask)
            d_emb = model(d_ids.view(B * D, L), d_mask.view(B * D, L)).view(B, D, -1)
            student_scores = torch.einsum("bh,bdh->bd", q_emb, d_emb)
            loss = kd_loss(student_scores, gold, temperature)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total += loss.item()
        n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total / max(n, 1)


# ── Eval loops ───────────────────────────────────────────────────────────────

@torch.no_grad()
def eval_mnrl_epoch(
    model,
    loader,
    device,
    temperature,
    use_amp,
):
    model.eval()
    total, n = 0.0, 0
    for batch in tqdm(loader, desc="eval", leave=False):
        batch = _move(batch, device)
        q_ids, q_mask = batch["q_ids"], batch["q_mask"]
        p_ids, p_mask = batch["p_ids"], batch["p_mask"]
        n_ids, n_mask = batch["n_ids"], batch["n_mask"]
        B, N, L = n_ids.shape
        with torch.autocast(device_type="cuda", enabled=use_amp, dtype=torch.bfloat16):
            q_emb = model(q_ids, q_mask)
            p_emb = model(p_ids, p_mask)
            n_flat = model(n_ids.view(B * N, L), n_mask.view(B * N, L)).view(B, N, -1)
            loss = mnrl_loss(q_emb, p_emb, n_flat, temperature)
        total += loss.item()
        n += 1
    return total / max(n, 1)


@torch.no_grad()
def eval_kd_epoch(
    model,
    loader,
    device,
    temperature,
    use_amp,
):
    model.eval()
    total, n = 0.0, 0
    for batch in tqdm(loader, desc="eval", leave=False):
        batch = _move(batch, device)
        q_ids, q_mask = batch["q_ids"], batch["q_mask"]
        d_ids, d_mask = batch["d_ids"], batch["d_mask"]
        gold = batch["gold_scores"]
        B, D, L = d_ids.shape
        with torch.autocast(device_type="cuda", enabled=use_amp, dtype=torch.bfloat16):
            q_emb = model(q_ids, q_mask)
            d_emb = model(d_ids.view(B * D, L), d_mask.view(B * D, L)).view(B, D, -1)
            student_scores = torch.einsum("bh,bdh->bd", q_emb, d_emb)
            loss = kd_loss(student_scores, gold, temperature)
        total += loss.item()
        n += 1
    return total / max(n, 1)


## Helper functions

In [ ]:
import json
import logging
import math
import random

import numpy as np

logging.basicConfig(
    force=True,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)


# ── Triplet derivation (used when hard_neg files are missing) ───────────────
# The scored data is the ground truth; triplets are a coarse binarisation so the
# same dataset can drive hard_neg (MNRL) training without a separate pass.
def derive_triplets_from_scored(scored_path, output_dir, max_negatives=4):
    from pathlib import Path
    stem = Path(scored_path).stem
    triplet_path = Path(output_dir) / f"{stem}_triplets.jsonl"
    pairs_path   = Path(output_dir) / f"{stem}_pairs.jsonl"
    n_triplets = n_pairs = 0
    with (
        open(triplet_path, "w", encoding="utf-8") as tf,
        open(pairs_path,   "w", encoding="utf-8") as pf,
    ):
        for raw in Path(scored_path).read_text(encoding="utf-8").splitlines():
            if not raw.strip():
                continue
            rec = json.loads(raw)
            docs = rec.get("docs", [])
            if len(docs) < 2:
                continue
            sorted_docs = sorted(docs, key=lambda d: d["teacher_score"], reverse=True)
            positive  = sorted_docs[0]["text"]
            negatives = [d["text"] for d in sorted_docs[-max_negatives:]]
            base = {"query": rec["query"], "persona_id": rec.get("persona_id", "")}
            tf.write(json.dumps({**base, "positive": positive, "negatives": negatives}, ensure_ascii=False) + "\n")
            n_triplets += 1
            for neg in negatives:
                pf.write(json.dumps({**base, "positive": positive, "negative": neg}, ensure_ascii=False) + "\n")
                n_pairs += 1
    logger.info("Derived %d triplets and %d pairs \u2192 %s, %s", n_triplets, n_pairs, triplet_path.name, pairs_path.name)


# ── Eval data loading ────────────────────────────────────────────────────────
def load_triplets_for_eval(path: str):
    """Load only query and positive from a triplets/pairs JSONL, applying persona if present."""
    items = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line.strip())
            query = obj["query"]
            if "persona_id" in obj:
                persona_text = render_profile(obj["persona_id"])
                query = f"{persona_text}\n\n{query}"
            items.append(
                {
                    "query": query,
                    "positive": obj["positive"],
                }
            )
    return items


def load_scored_for_eval(path: str):
    """Group scored rows by query - each query has multiple (document, score) pairs.
    Applies persona to the query if present.
    """
    from collections import defaultdict

    grouped = defaultdict(list)
    order = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line.strip())
            q = obj["query"]
            if "persona_id" in obj:
                persona_text = render_profile(obj["persona_id"])
                q = f"{persona_text}\n\n{q}"
            if q not in grouped:
                order.append(q)
            if "docs" in obj:
                for doc in obj["docs"]:
                    grouped[q].append((doc["text"], float(doc["teacher_score"])))
            elif "document" in obj and "score" in obj:
                grouped[q].append((obj["document"], float(obj["score"])))
            else:
                continue
    return [
        {
            "query": q,
            "documents": [d for d, _ in grouped[q]],
            "scores": [s for _, s in grouped[q]],
        }
        for q in order
    ]


# ── Encoding for eval ────────────────────────────────────────────────────────
@torch.no_grad()
def encode_texts(
    model,
    tokenizer,
    texts,
    device,
    batch_size: int = 32,
    max_length: int = 512,
):
    """Encode texts using the trained wrapper model (mean-pool + L2)."""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        tokens = tokenizer(
            batch,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        ).to(device)
        emb = model(tokens["input_ids"], tokens["attention_mask"])
        all_embs.append(emb.cpu().float().numpy())
    return np.vstack(all_embs)


# ── Metrics ──────────────────────────────────────────────────────────────────
def recall_at_k(scores, top_k):
    """For each row i, the ground-truth index is also i."""
    Q = scores.shape[0]
    results = {}
    for k in range(1, top_k + 1):
        hits = 0
        for i in range(Q):
            topk_idx = np.argsort(scores[i])[::-1][:k]
            if i in topk_idx:
                hits += 1
        results[f"recall@{k}"] = hits / max(Q, 1)
    return results


def mrr(scores):
    Q = scores.shape[0]
    total = 0.0
    for i in range(Q):
        ranked = np.argsort(scores[i])[::-1]
        rank_pos = np.where(ranked == i)[0]
        if len(rank_pos) > 0:
            total += 1.0 / (rank_pos[0] + 1)
    return total / max(Q, 1)


def ndcg_at_k(gold_ranks, k):
    """NDCG@k for a single query (higher gold rank = more relevant)."""
    dcg = 0.0
    for i, g in enumerate(gold_ranks[:k]):
        dcg += g / math.log2(i + 2)
    ideal = sorted(gold_ranks, reverse=True)[:k]
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0


# ── Full eval routine ────────────────────────────────────────────────────────
def evaluate_retrieval(
    model,
    tokenizer,
    test_path,
    fmt,
    device,
    top_k,
    batch_size=32,
):
    """Compute retrieval metrics on the validation set.

    Args:
        fmt: "triplets", "pairs", or "scored"
    """
    if fmt in ("triplets", "pairs"):
        items = load_triplets_for_eval(test_path)
        queries = [it["query"] for it in items]
        positives = [it["positive"] for it in items]

        q_embs = encode_texts(model, tokenizer, queries, device, batch_size)
        p_embs = encode_texts(model, tokenizer, positives, device, batch_size)
        scores = q_embs @ p_embs.T  # [Q, P]

        results = recall_at_k(scores, top_k)
        results["mrr"] = mrr(scores)
        return results

    else:  # scored
        items = load_scored_for_eval(test_path)
        recall_hits = {k: 0 for k in range(1, top_k + 1)}
        ndcg_sum = 0.0
        n = 0

        for it in items:
            q = it["query"]
            docs = it["documents"]
            gold = it["scores"]
            if len(docs) < 2:
                continue

            q_emb = encode_texts(model, tokenizer, [q], device, batch_size)  # [1, H]
            d_emb = encode_texts(model, tokenizer, docs, device, batch_size)  # [D, H]
            sims = (q_emb @ d_emb.T)[0]  # [D]

            max_gold = max(gold)
            positive_indices = [i for i, s in enumerate(gold) if abs(s - max_gold) < 1e-6]

            ranked = np.argsort(sims)[::-1]
            for k in range(1, top_k + 1):
                topk_idx = ranked[:k]
                if any(p in topk_idx for p in positive_indices):
                    recall_hits[k] += 1

            gold_arr = np.array(gold, dtype=float)
            gold_ranks = gold_arr[ranked].tolist()
            ndcg_sum += ndcg_at_k(gold_ranks, top_k)
            n += 1

        results = {}
        for k in range(1, top_k + 1):
            results[f"recall@{k}"] = recall_hits[k] / max(n, 1)
        results[f"ndcg@{top_k}"] = ndcg_sum / max(n, 1)
        return results


## Run pipeline

In [ ]:
import math
import random
import sys

import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

seed = CFG["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = CFG["training"]["device"]
mode = CFG.get("mode", "reader_kd")
fmt  = CFG.get("format", "triplets")
precision = CFG["training"]["precision"]
use_amp   = device == "cuda" and precision in ("fp16", "bf16")

model_name  = CFG["embedder"]["model"]
max_length  = CFG["embedder"]["max_seq_length"]
temperature = CFG["training"]["temperature"]
max_negatives = CFG["training"].get("max_negatives", 4)
max_documents = CFG["training"].get("max_documents", 20)
eval_top_k    = CFG["eval"]["top_k"]
eval_batch_sz = CFG["eval"]["eval_batch_size"]
best_metric   = CFG.get("best_metric")
grad_clip     = CFG["training"].get("grad_clip", 1.0)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Build datasets and collators based on mode
_data_dir = Path(CFG["data"]["train_data"])
if mode == "reader_kd":
    train_path = _data_dir / "train.jsonl"
    val_path   = _data_dir / "val.jsonl"
elif fmt == "triplets":
    train_path = _data_dir / "train_triplets.jsonl"
    val_path   = _data_dir / "val_triplets.jsonl"
else:  # pairs
    train_path = _data_dir / "train_pairs.jsonl"
    val_path   = _data_dir / "val_pairs.jsonl"

# Auto-derive triplets/pairs if hard_neg files are missing.
if mode == "hard_neg" and not train_path.exists():
    _scored_train = train_path.parent / "train.jsonl"
    if not _scored_train.exists():
        raise FileNotFoundError(
            f"{train_path.name} not found and no train.jsonl to derive from. "
            "Run gen_ropg_data first."
        )
    logger.warning("%s not found — deriving triplets/pairs from scored data.", train_path.name)
    _data_dir = train_path.parent
    for _sp in (_data_dir / "train.jsonl", _data_dir / "val.jsonl"):
        if _sp.exists():
            derive_triplets_from_scored(_sp, _data_dir, max_negatives)

if mode == "hard_neg":
    if fmt == "triplets":
        train_ds = TripletDataset(str(train_path), max_negatives)
        val_ds   = TripletDataset(str(val_path), max_negatives) if val_path.exists() else None
        collate_fn = lambda b: collate_triplets(b, tokenizer, max_length)
        eval_fmt = "triplets"
    else:
        train_ds = PairDataset(str(train_path))
        val_ds   = PairDataset(str(val_path)) if val_path.exists() else None
        collate_fn = lambda b: collate_pairs(b, tokenizer, max_length)
        eval_fmt = "pairs"
else:  # reader_kd
    train_ds = ScoredDataset(str(train_path), max_documents)
    val_ds   = ScoredDataset(str(val_path), max_documents) if val_path.exists() else None
    collate_fn = lambda b: collate_scored(b, tokenizer, max_length)
    eval_fmt = "scored"

_num_workers = 2 if sys.platform != "darwin" else 0
train_loader = DataLoader(
    train_ds, batch_size=CFG["training"]["batch_size"], shuffle=True,
    num_workers=_num_workers, collate_fn=collate_fn, drop_last=True,
)
val_loader = (
    DataLoader(val_ds, batch_size=CFG["training"]["batch_size"], shuffle=False,
               num_workers=_num_workers, collate_fn=collate_fn, drop_last=False)
    if val_ds else None
)

model = QwenEmbeddingModel(
    model_name,
    use_gradient_checkpointing=CFG["training"].get("gradient_checkpointing", True),
).to(device)
logger.info("Parameters: %d", sum(p.numel() for p in model.parameters()))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["training"]["lr"],
    weight_decay=CFG["training"].get("weight_decay", 0.01),
)
n_epochs        = CFG["training"]["epochs"]
n_steps_per_ep  = max(1, math.ceil(len(train_ds) / CFG["training"]["batch_size"]))
total_steps     = n_steps_per_ep * n_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * CFG["training"].get("warmup_ratio", 0.1)),
    num_training_steps=total_steps,
)

train_fn    = train_mnrl_epoch if mode == "hard_neg" else train_kd_epoch
eval_loss_fn = eval_mnrl_epoch if mode == "hard_neg" else eval_kd_epoch

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()


In [ ]:
# Quick sanity check: one forward + backward to catch shape/NaN problems early.
import contextlib

sample = train_ds[0]
if mode == "hard_neg":
    batch = collate_triplets([sample], tokenizer, max_length)
else:
    batch = collate_scored([sample], tokenizer, max_length)
batch = _move(batch, torch.device(device))

model.train()
ctx = torch.autocast("cuda", dtype=torch.float16) if use_amp else contextlib.nullcontext()
with ctx:
    if mode == "hard_neg":
        q_emb = model(batch["q_ids"], batch["q_mask"])
        p_emb = model(batch["p_ids"], batch["p_mask"])
        B, N, L = batch["n_ids"].shape
        n_flat = model(batch["n_ids"].view(B * N, L), batch["n_mask"].view(B * N, L)).view(B, N, -1)
        health_loss = mnrl_loss(q_emb, p_emb, n_flat, temperature)
    else:
        q_emb = model(batch["q_ids"], batch["q_mask"])
        B, D, L = batch["d_ids"].shape
        d_emb = model(batch["d_ids"].view(B * D, L), batch["d_mask"].view(B * D, L)).view(B, D, -1)
        student_scores = torch.einsum("bh,bdh->bd", q_emb, d_emb)
        health_loss = kd_loss(student_scores, batch["gold_scores"], temperature)

assert torch.isfinite(health_loss), f"Health check failed: loss={health_loss}"
print(f"Health check loss: {health_loss.item():.4f}")
health_loss.backward()
optimizer.zero_grad()
if device == "cuda":
    print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()


In [ ]:
import contextlib
from pathlib import Path
from tqdm.auto import tqdm

checkpoint_dir = Path(CFG["checkpoint_dir"])
checkpoint_dir.mkdir(parents=True, exist_ok=True)
save_dir = checkpoint_dir / "checkpoint-best"

best_metric_name = best_metric if best_metric else "val_loss"
best_metric_goal = "max" if best_metric else "min"
best_val_loss    = float("inf")
best_metric_val  = -float("inf") if best_metric_goal == "max" else float("inf")
best_epoch       = 0
epoch_metrics    = []

for epoch in tqdm(range(1, n_epochs + 1), desc="epochs"):
    train_loss = train_fn(model, train_loader, optimizer, scheduler, torch.device(device), temperature, use_amp)

    val_str = ""
    entry   = {"epoch": epoch, "train_loss": train_loss}

    if val_loader and val_path.exists():
        val_loss = eval_loss_fn(model, val_loader, torch.device(device), temperature, use_amp)
        val_str  = f" | Val Loss: {val_loss:.4f}"
        entry["val_loss"] = val_loss

        eval_metrics = evaluate_retrieval(model, tokenizer, str(val_path), eval_fmt, device, eval_top_k, eval_batch_sz)
        val_str += " | " + " | ".join(f"{k}: {v:.4f}" for k, v in eval_metrics.items())
        entry.update(eval_metrics)

        if best_metric is None:
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                model.model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)
                best_epoch = epoch
                val_str += " *best (loss)*"
        else:
            cur = eval_metrics.get(best_metric_name)
            if cur is not None:
                improved = cur > best_metric_val if best_metric_goal == "max" else cur < best_metric_val
                if improved:
                    best_metric_val = cur
                    model.model.save_pretrained(save_dir)
                    tokenizer.save_pretrained(save_dir)
                    best_epoch = epoch
                    val_str += f" *best ({best_metric_name}: {cur:.4f})*"
            else:
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    model.model.save_pretrained(save_dir)
                    tokenizer.save_pretrained(save_dir)
                    best_epoch = epoch
                    val_str += " *best (loss fallback)*"

    epoch_metrics.append(entry)
    logger.info("Epoch %d/%d — Train Loss: %.4f%s", epoch, n_epochs, train_loss, val_str)

    if device == "cuda":
        logger.info("CUDA peak | alloc: %.2f GB | reserved: %.2f GB",
                    torch.cuda.max_memory_allocated() / 1e9,
                    torch.cuda.max_memory_reserved() / 1e9)
        torch.cuda.empty_cache()

    freq = max(1, n_epochs // 3)
    if epoch % freq == 0:
        ckpt = checkpoint_dir / f"checkpoint-epoch-{epoch}"
        model.model.save_pretrained(ckpt)
        tokenizer.save_pretrained(ckpt)
        logger.info("Periodic checkpoint: %s", ckpt)

final_dir = checkpoint_dir / "checkpoint-final"
model.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)

import json
(checkpoint_dir / "training_log.json").write_text(
    json.dumps({"config": CFG, "seed": seed, "epoch_metrics": epoch_metrics, "best_epoch": best_epoch},
               ensure_ascii=False, indent=2),
    encoding="utf-8",
)
logger.info("Best epoch: %d | saved to %s", best_epoch, save_dir)
logger.info("Final checkpoint: %s", final_dir)


## Download

In [ ]:
import subprocess
from pathlib import Path

if RUNTIME != "colab":
    subprocess.run(
        [
            "zip", "-r", "ropg_kd_model.zip",
            "ropg_kd_checkpoints/checkpoint-best",
            "ropg_kd_checkpoints/training_log.json",
        ],
        cwd=str(Path(OUTPUT_DIR).parent),
        check=True,
    )
    print("ropg_kd_adapter.zip created.")
else:
    print("Colab: checkpoints already on Google Drive — skipping zip.")